# Parameter Sweep Demo for Swing Range Expansion Strategy

This notebook demonstrates how to use the parameter sweep utility to optimize strategy parameters and visualize the results.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

try:
    from utils.experiments.sweeper import run_parameter_sweep
    from src.strategies.swing_range_expansion.runner.backtest_runner import SwingRangeExpansionBacktestRunner
    print("✅ All imports successful!")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Please ensure you're running from the correct directory and all modules are available.")

## Setup Parameter Grid

Define the parameter ranges to sweep over:

In [ ]:
# Define parameter grid for sweep - using correct SwingRangeConfig parameter names
param_grid = {
    'nr_lookback': [5, 7, 10],           # Days to test for narrowest range
    'target_rr': [1.0, 1.5, 2.0],       # × NR range for take-profit
    'stop_rr': [0.5, 0.75, 1.0],        # × NR range for stop-loss
}

print(f"Total combinations: {len(param_grid['nr_lookback']) * len(param_grid['target_rr']) * len(param_grid['stop_rr'])}")
print(f"Parameter grid: {param_grid}")

## Run Parameter Sweep

Execute the parameter sweep using the utility:

In [ ]:
# Run parameter sweep
print("Starting parameter sweep...")
instrument_ids = ["NIFTY.D.NSE"]

try:
    results_df = run_parameter_sweep(
        SwingRangeExpansionBacktestRunner, 
        param_grid, 
        instrument_ids,
        workers=4
    )
    
    print(f"Completed {len(results_df)} parameter combinations")
    print("\nFirst few results:")
    print(results_df.head())
    
    if 'error' in results_df.columns:
        error_count = results_df['error'].notna().sum()
        if error_count > 0:
            print(f"\n⚠️  {error_count} runs had errors")
            
except Exception as e:
    print(f"❌ Error: {e}")
    print("Creating dummy data for demonstration...")
    np.random.seed(42)
    dummy_data = []
    for nr in param_grid['nr_lookback']:
        for target in param_grid['target_rr']:
            for stop in param_grid['stop_rr']:
                dummy_data.append({
                    'nr_lookback': nr,
                    'target_rr': target,
                    'stop_rr': stop,
                    'return_pct': np.random.normal(0.1, 0.05),
                    'sharpe': np.random.normal(1.2, 0.3),
                    'mdd_pct': np.random.normal(-0.15, 0.05)
                })
    results_df = pd.DataFrame(dummy_data)
    print(f"Created {len(results_df)} dummy results")

## Analyze Results

In [ ]:
# Display basic statistics
print("Parameter Sweep Results Summary:")
print(f"Number of combinations: {len(results_df)}")

return_col = 'return_pct' if 'return_pct' in results_df.columns else 'total_return'
sharpe_col = 'sharpe' if 'sharpe' in results_df.columns else 'sharpe_ratio'

if return_col in results_df.columns:
    print(f"Best Return: {results_df[return_col].max():.2%}")
    print(f"Average Return: {results_df[return_col].mean():.2%}")

if sharpe_col in results_df.columns:
    print(f"Best Sharpe: {results_df[sharpe_col].max():.3f}")

if 'mdd_pct' in results_df.columns:
    print(f"Lowest MDD: {results_df['mdd_pct'].min():.2%}")

# Top performers
if return_col in results_df.columns:
    print(f"\nTop 5 by {return_col}:")
    cols = ['nr_lookback', 'target_rr', 'stop_rr', return_col]
    if sharpe_col in results_df.columns:
        cols.append(sharpe_col)
    print(results_df.nlargest(5, return_col)[cols])

## 3D Visualization

In [ ]:
# 3D scatter plot
return_col = 'return_pct' if 'return_pct' in results_df.columns else 'total_return'
sharpe_col = 'sharpe' if 'sharpe' in results_df.columns else 'sharpe_ratio'

required_cols = ['nr_lookback', 'target_rr', 'stop_rr', return_col]
if all(col in results_df.columns for col in required_cols):
    fig = go.Figure(data=go.Scatter3d(
        x=results_df['nr_lookback'],
        y=results_df['target_rr'],
        z=results_df['stop_rr'],
        mode='markers',
        marker=dict(
            size=8,
            color=results_df[return_col],
            colorscale='Viridis',
            colorbar=dict(title="Return %"),
            showscale=True
        ),
        text=[f"NR: {nr}<br>Target: {target:.1f}R<br>Stop: {stop:.1f}R<br>Return: {ret:.2%}"
              for nr, target, stop, ret in zip(
                  results_df['nr_lookback'], 
                  results_df['target_rr'], 
                  results_df['stop_rr'], 
                  results_df[return_col]
              )],
        hovertemplate='%{text}<extra></extra>'
    ))
    
    fig.update_layout(
        title='Parameter Sweep Results',
        scene=dict(
            xaxis_title='NR Lookback (Days)',
            yaxis_title='Target RR',
            zaxis_title='Stop RR'
        ),
        width=800,
        height=600
    )
    
    fig.show()
else:
    print(f"❌ Missing columns for visualization")
    print(f"Available: {list(results_df.columns)}")

## Risk-Return Analysis

In [ ]:
# Risk-Return scatter plot
return_col = 'return_pct' if 'return_pct' in results_df.columns else 'total_return'
sharpe_col = 'sharpe' if 'sharpe' in results_df.columns else 'sharpe_ratio'

if all(col in results_df.columns for col in [return_col, 'mdd_pct']):
    color_col = sharpe_col if sharpe_col in results_df.columns else None
    
    fig = px.scatter(
        results_df, 
        x='mdd_pct', 
        y=return_col,
        color=color_col,
        hover_data=['nr_lookback', 'target_rr', 'stop_rr'],
        title='Risk-Return Profile',
        labels={
            'mdd_pct': 'Maximum Drawdown (%)',
            return_col: 'Return (%)'
        }
    )
    
    fig.update_layout(width=800, height=600)
    fig.show()
    
    # Best risk-adjusted performance
    if sharpe_col in results_df.columns:
        best_sharpe = results_df.loc[results_df[sharpe_col].idxmax()]
        print(f"\nBest Risk-Adjusted Performance:")
        print(f"NR Lookback: {best_sharpe['nr_lookback']}, Target: {best_sharpe['target_rr']:.1f}R, Stop: {best_sharpe['stop_rr']:.1f}R")
        print(f"Return: {best_sharpe[return_col]:.2%}, Sharpe: {best_sharpe[sharpe_col]:.3f}")
else:
    print(f"❌ Missing columns for risk-return analysis")
    print(f"Available: {list(results_df.columns)}")